# 01 · Signaling

The first of the four themes, and the template the other three follow.

**Panel:** Foxo3a, Foxo1, FGFR1, PDGFRα, β-Catenin, YAP1, p-S6, p-AKT, c-Myc, RNAPII-pS5

This theme has something the others do not: a **prediction**. Five of the eighteen
compounds target the PI3K–AKT–mTOR axis, so we know in advance roughly what should
move. That makes signaling the right place to learn what a real effect looks like in
this dataset — and what a real *absence* looks like.

Five steps, repeated in every theme chapter:

1. select the panel
2. per-condition effects, as a heatmap
3. rank the effects, honestly
4. an embedding built from this theme alone
5. what happens to single cells over time

In [ ]:
from math import comb
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats

plt.rcParams.update({          # the house style, no package needed
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "legend.frameon": False, "pdf.fonttype": 42, "ps.fonttype": 42,
})

def panel_grid(n, *, ncols=3, size=(3.6, 3.0)):
    """A figure with `n` axes on a grid, the unused ones removed."""
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, squeeze=False,
                             figsize=(size[0] * ncols, size[1] * nrows))
    flat = axes.ravel()
    for ax in flat[n:]:
        ax.remove()
    return fig, flat[:n]
pd.set_option("display.width", 140)

# The tables live here on Euler. Change this line if your copy is elsewhere.
DATA = Path("/cluster/project/mcsliberali/data_mcs_2026")

THEME = "signaling"

cells = sc.read_h5ad(DATA / "mcs2026_intensity.h5ad")
wells = (cells.to_df()
         .groupby([cells.obs.condition.astype(str), cells.obs.timepoint_h.astype(int),
                   cells.obs.well.astype(str)], observed=True).mean()
         .rename_axis(["condition", "timepoint", "well"]).reset_index())
print(f"{len(wells)} wells, {cells.n_obs:,} cells")

def effect_table(wells, markers, control="DMSO", by_timepoint=True):
    """Mean shift from the control, per condition and marker, in control SDs.

    The control is subtracted explicitly. Skipping it looks safe -- the
    normalisation puts the controls near zero -- but that only holds at the one
    timepoint the origin came from; everywhere else a plain group mean would
    report the controls' own development as a treatment effect.
    """
    if by_timepoint:
        table = wells.groupby(["condition", "timepoint"], observed=True)[markers].mean()
        reference = (wells[wells.condition == control]
                     .groupby("timepoint", observed=True)[markers].mean())
        matched = reference.reindex(table.index.get_level_values("timepoint"))
        return (table - matched.to_numpy()).drop(index=control, level=0, errors="ignore")
    table = wells.groupby("condition", observed=True)[markers].mean()
    return (table - wells.loc[wells.condition == control, markers].mean()
            ).drop(index=control, errors="ignore")

def rank_effects(wells, markers, control="DMSO", drop=None):
    """Every condition x marker pair, ranked by absolute shift.

    `p_floor` is the smallest p-value this design can produce: with n treated
    and m control wells there are C(n+m, n) orderings, so a two-sided
    Mann-Whitney can never go below 2/C(n+m, n). A row sitting at the floor is
    not more significant than another row at the floor, whatever its effect size.
    """
    frame = wells if drop is None else wells[wells.condition != drop]
    reference = frame[frame.condition == control]
    rows = []
    for condition, block in frame.groupby("condition", observed=True):
        if condition == control:
            continue
        for marker in markers:
            treated, base = block[marker].dropna(), reference[marker].dropna()
            if len(treated) < 3 or len(base) < 3:
                continue
            rows.append({"condition": condition, "marker": marker,
                         "shift": treated.mean() - base.mean(), "n_wells": len(treated),
                         "p": stats.mannwhitneyu(treated, base).pvalue,
                         "p_floor": 2 / comb(len(treated) + len(base), len(treated))})
    out = pd.DataFrame(rows)
    out["abs_shift"] = out["shift"].abs()
    out["at_p_floor"] = np.isclose(out["p"], out["p_floor"])
    return out.sort_values("abs_shift", ascending=False).reset_index(drop=True)

def compare_to_control(wells, marker, condition, control="DMSO"):
    """One marker, one condition, per timepoint, against the control wells."""
    rows = []
    for timepoint, block in wells.groupby("timepoint", observed=True):
        treated = block.loc[block.condition == condition, marker].dropna()
        base = block.loc[block.condition == control, marker].dropna()
        if len(treated) < 2 or len(base) < 2:
            continue
        rows.append({"timepoint": timepoint, "n_treated": len(treated),
                     "n_control": len(base),
                     "shift": round(treated.mean() - base.mean(), 2),
                     "p": round(stats.mannwhitneyu(treated, base).pvalue, 4),
                     "p_floor": round(2 / comb(len(treated) + len(base), len(treated)), 4)})
    return pd.DataFrame(rows)

OUTLIER = cells.obs.condition[cells.obs.is_outlier_condition].astype(str).iloc[0]


## 1. The panel

In [ ]:
markers = [m for m in cells.uns["panels"][THEME] if m in set(wells.columns)]
print(f"{len(markers)} of {len(cells.uns['panels'][THEME])} panel markers present:")
markers

Where each was imaged matters, because Step 15 showed that signal level depends on
the round. The normalisation handles it, but it is worth seeing which markers came from
the fragile late rounds:

In [ ]:
lookup = cells.var[cells.var.marker.isin(markers) & (cells.var.statistic == "mean_intensity")]
lookup[["marker", "round", "channel", "intensity_threshold"]].sort_values("round").reset_index(drop=True)

## 2. Effects per condition

Every value below is in **control-well standard deviations**: how far a condition sits
from the DMSO wells, in units of how much two control wells differ from each other.
`effect_table` subtracts the controls explicitly, so what is left is the treatment and
not the controls' own development across the time course.

In [ ]:
effects = effect_table(wells, markers, by_timepoint=False)
effects.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
order = effects.abs().mean(axis=1).sort_values(ascending=False).index
matrix = effects.loc[order]
limit = np.nanpercentile(np.abs(matrix.values), 99)
im = ax.imshow(matrix.values, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
ax.set(xticks=range(len(markers)), yticks=range(len(order)), yticklabels=order)
ax.set_xticks(range(len(markers)))
ax.set_xticklabels(markers, rotation=45, ha="right")
ax.set_title(f"{THEME.capitalize()} panel — shift from DMSO (control SDs)")
fig.colorbar(im, ax=ax, shrink=0.7, label="control SDs")
fig.tight_layout()

Two things jump out, and only one of them is about signaling.

**PMA is the whole top row.** As [Step 16](../part3_analysis/1_preparation/03_normalisation.ipynb) showed, it moves everything. It is a real
effect and it is not a *signaling-specific* effect, so from here on we set it aside and
look at the other seventeen. It comes back in [chapter 05](05_integration.ipynb).

**Below PMA, the pattern is structured**, not noise: particular conditions move
particular markers, and the same markers move together.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
without_outlier = effects.drop(index=OUTLIER, errors="ignore")
order = without_outlier.abs().mean(axis=1).sort_values(ascending=False).index
matrix = without_outlier.loc[order]
limit = np.nanpercentile(np.abs(matrix.values), 99)
im = ax.imshow(matrix.values, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
ax.set(yticks=range(len(order)), yticklabels=order)
ax.set_xticks(range(len(markers)))
ax.set_xticklabels(markers, rotation=45, ha="right")
ax.set_title(f"{THEME.capitalize()} panel, PMA excluded")
fig.colorbar(im, ax=ax, shrink=0.7, label="control SDs")
fig.tight_layout()

## 3. Rank the effects — honestly

In [ ]:
ranked = rank_effects(wells, markers, drop=OUTLIER)
ranked.head(15).round(3)

:::{warning}
Look at the `p` and `p_floor` columns together. With 3 treated wells against 20 control
wells, the smallest achievable p-value is fixed by the design; `at_p_floor` marks the
rows that have hit it. A row at the floor is **not** more significant than another row
at the floor, no matter how different their effect sizes are.

This is why the column that should drive your reading is `shift`, in control SDs, and
not `p`.
:::

In [ ]:
print(f"{ranked.at_p_floor.sum()} of {len(ranked)} comparisons sit exactly at the p-value floor")
ranked.groupby("condition")["abs_shift"].mean().sort_values(ascending=False).head(8).round(2)

### The prediction

Sapanisertib/INK128 inhibits mTOR. mTORC1 activates S6 kinase, which phosphorylates
ribosomal protein S6. So p-S6 should fall.

In [ ]:
compare_to_control(wells, "p-S6", "Sapanisertib/INK128")

In [ ]:
compare_to_control(wells, "p-S6", "MK-2206")

It does — several control SDs down at 36, 48 and 60 hours, for both the mTOR inhibitor
and the AKT inhibitor upstream of it. And it fades by 84 h, which is what you would
expect from a compound that is consumed or adapted to.

### The prediction that fails

The obvious readout for an AKT inhibitor is phospho-AKT.

In [ ]:
pd.concat(
    [compare_to_control(wells, "p-AKT", c).assign(condition=c)
     for c in ["MK-2206", "Wortmannin", "Sapanisertib/INK128"]]
)[["condition", "timepoint", "shift", "p", "p_floor"]]

Nothing moves. This is the single most useful result in the chapter, because the
temptation is to assume the experiment failed. Three reasons it has not:

- **Feedback.** Blocking mTORC1 relieves a negative feedback loop onto the receptor,
  which *raises* AKT phosphorylation. The drug pushes down, the feedback pushes up.
- **Epitope fragility.** p-AKT sits in round 24, near the end of 18 elution cycles.
  Phospho-epitopes are the first thing a 4i panel loses.
- **Downstream integrates.** p-S6 reflects sustained pathway output; a single
  phospho-site is a snapshot of a fast equilibrium.

The pathway *is* visible — one step sideways:

In [ ]:
pd.concat(
    [compare_to_control(wells, "Foxo1", c).assign(condition=c)
     for c in ["MK-2206", "Wortmannin", "Sapanisertib/INK128"]]
)[["condition", "timepoint", "shift", "p"]]

**Foxo1 falls for all three inhibitors, at every timepoint.** FoxO transcription factors
are direct AKT substrates. This is the pathway signature: not in the phospho-antibody
you would have picked first, but in a downstream target and a downstream effector.

## 4. An embedding of this theme alone

A PCA over *all* 2,587 features answers the question "how do these wells differ in any
way at all". A PCA over ten signaling markers answers "how do these wells differ in
their signaling" — a smaller question with a readable answer.

In [ ]:
usable = wells[wells.condition != OUTLIER]
space = ad.AnnData(usable[markers].fillna(0).to_numpy(dtype="float32"))
sc.pp.pca(space, n_comps=4, random_state=0)
coords, variance = space.obsm["X_pca"], space.uns["pca"]["variance_ratio"]
print("variance explained:", variance[:4].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
timepoints = usable.timepoint.values
scatter = axes[0].scatter(coords[:, 0], coords[:, 1], c=timepoints, cmap="viridis", s=28)
axes[0].set(xlabel=f"PC1 ({variance[0]:.0%})",
            ylabel=f"PC2 ({variance[1]:.0%})",
            title="Signaling space, coloured by timepoint")
fig.colorbar(scatter, ax=axes[0], label="hours")

highlight = {"DMSO": "0.6", "Sapanisertib/INK128": "firebrick",
             "MK-2206": "darkorange", "RA": "seagreen"}
axes[1].scatter(coords[:, 0], coords[:, 1], c="0.88", s=22)
for condition, colour in highlight.items():
    mask = (usable.condition == condition).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1], c=colour, s=34, label=condition)
axes[1].set(xlabel="PC1", ylabel="PC2", title="Four conditions picked out")
axes[1].legend(fontsize=7)
fig.tight_layout()

In [ ]:
loadings = pd.DataFrame(space.varm["PCs"][:, :2], index=markers,
                        columns=["PC1", "PC2"])
loadings.reindex(loadings.PC1.abs().sort_values(ascending=False).index).round(2)

The loadings say what the axes mean. Read them before interpreting any position in the
plot — a point being "to the right" is only informative once you know what PC1 is made
of.

## 5. Single cells, not just wells

Everything above collapsed each well to a mean. That is right for *testing*, because
the well is the replicate. But it throws away the thing imaging gives you that a plate
reader does not: the distribution.

A shift in the mean can happen two ways — every cell moves a little, or a subpopulation
moves a lot — and those are different biology.

In [ ]:
def marker_column(marker: str) -> str:
    match = cells.var[(cells.var.marker == marker)
                      & (cells.var.statistic == "mean_intensity")
                      & (cells.var.family == "Intensity")]
    return match.index[0]

column = marker_column("p-S6")
frame = pd.DataFrame({
    "value": np.asarray(cells[:, column].X).ravel(),
    "condition": cells.obs.condition.astype(str).values,
    "timepoint": cells.obs.timepoint_h.astype(int).values,
})

fig, axes = panel_grid(4, ncols=4, size=(3.3, 2.9))
for ax, timepoint in zip(axes, [36, 48, 60, 84]):
    block = frame[frame.timepoint == timepoint]
    for condition, colour in [("DMSO", "0.35"), ("Sapanisertib/INK128", "firebrick")]:
        values = block.loc[block.condition == condition, "value"]
        ax.hist(values, bins=60, histtype="step", density=True,
                color=colour, linewidth=1.4, label=condition)
    ax.set(title=f"{timepoint} h", xlabel="p-S6 (control-cell SDs)")
    if timepoint == 36:
        ax.set_ylabel("density")
        ax.legend(fontsize=6)
fig.tight_layout()

The whole distribution shifts left at 36–60 h, rather than a subpopulation splitting
off. So mTOR inhibition here is a **uniform** effect on the population: every cell
lowers its p-S6, rather than some fraction of cells switching off.

That distinction is invisible in the well means, and it is the reason to look.

## 6. One number for the theme

For [chapter 05](05_integration.ipynb) we need a single summary per condition, so the four themes can be
compared:

In [ ]:
summary = effect_table(wells, markers, by_timepoint=False).abs().mean(axis=1).sort_values(ascending=False)
summary.head(8).round(2)

In [ ]:
summary.to_frame(THEME).to_parquet(DATA / f"theme_{THEME}.parquet")
print(f"saved theme summary for {THEME}")

---

## Exercises

### 1. Which marker separates the timepoints?

PC1 above is coloured by timepoint and shows clear structure. Which markers load on it,
and does that make sense for cells differentiating over 36–84 hours?

:::{admonition} Solution
:class: dropdown

```python
loadings.PC1.sort_values()
```

Look for the pluripotency-associated and proliferation-associated markers. Cells left
in culture for 84 h are further along in differentiation than at 36 h, so markers
tracking that transition should dominate the axis that separates timepoints. If instead
PC1 loads mostly on one noisy marker, you are looking at a technical axis, and the
per-marker control spread from
[Step 17](../part3_analysis/1_preparation/03_normalisation.ipynb) will tell you which.
:::

### 2. Is IGF the mirror image of the inhibitors?

IGF activates the PI3K–AKT pathway; MK-2206 and INK128 inhibit it. If the panel is
reading the pathway, IGF should move the same markers in the opposite direction. Test
it on p-S6 and Foxo1. Does it?

:::{admonition} Solution
:class: dropdown

```python
for marker in ["p-S6", "Foxo1", "p-AKT"]:
    print(marker)
    print(compare_to_control(wells, marker, "IGF"))
```

The effect is much weaker than for the inhibitors, and that is worth thinking about
rather than dismissing. The cells are already growing in medium containing growth
factors, so the pathway is not off to begin with — there is far more room to inhibit a
active pathway than to further activate one. Asymmetry between activation and
inhibition is normal, and it is a reason to be careful about reading a weak effect as
"no pathway involvement".
:::

### 3. Repeat this chapter for another panel

Every step above is panel-agnostic. Change `THEME` to `"cell_cycle"` and re-run.
Which conditions move Cyclin A2, p21 and Ki67, and is that consistent with what those
compounds do?

:::{admonition} Solution
:class: dropdown

```python
THEME = "cell_cycle"
markers = [m for m in cells.uns["panels"][THEME] if m in set(wells.columns)]
rank_effects(wells, markers, drop=OUTLIER).head(10)
```

Expect Cycloheximide (translation block) and PMA to show up. Cyclin A2 marks S/G2 and
Ki67 marks any cycling cell, so a compound that arrests the cycle should lower both
while p21, a cyclin-dependent kinase inhibitor, may rise.

This is exactly what chapters 04–06 do with the other three panels — the code does not
change, only the biology you bring to reading it.
:::

---

**Next:** [02 · Cell-specific mechanics](02_mechanics.ipynb).